## 第 2 節｜隨機森林與 Grid Search

### 資料集欄位｜500 封模擬校園郵件

| 欄位 | 用途 |
|---|---|
| `email_id` | 郵件編號，不拿來訓練 |
| `link_count` | 網址連結數量，輸入 X |
| `urgent_words` | 緊急用語次數，輸入 X |
| `exclamation_count` | 驚嘆號次數，輸入 X |
| `sender_known` | 是否為已知寄件者（1 是、0 否），輸入 X |
| `email_type` | 郵件類型，透過 One-Hot 變成數值欄位 |
| `is_spam` | 是否為垃圾郵件（1 是、0 否），答案 y |

先將 `02_校園郵件分類_500筆.csv` 放在 Notebook 的工作目錄。第一個 Code Cell **直接執行**。


In [ ]:
# Import the dataset and model-building tools.
# 匯入讀檔、資料切分與模型工具。
import pandas as pd

# Prepare training splits and optional cross-validation.
# 準備資料切分與選做的交叉驗證。
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Load models for the main task and optional comparison.
# 載入隨機森林與選做的決策樹模型。
from sklearn.ensemble import RandomForestClassifier

# Load a single tree as the comparison model.
# 載入單棵決策樹作為比較模型。
from sklearn.tree import DecisionTreeClassifier

# Load the metric and optional chart library.
# 載入正確率與選做的圖表套件。
from sklearn.metrics import accuracy_score

# Enable optional graphs.
# 讓選做範例可以繪圖。
import matplotlib.pyplot as plt

# Read the example email dataset.
# 讀取 500 封模擬校園郵件。
df = pd.read_csv('02_校園郵件分類_500筆.csv')

# Preview the input data.
# 先查看前五封郵件。
display(df.head())


### 練習 1｜準備 X、y 與測試集（照範例輸入下一格）

```python
# Select only the features used to predict email categories.
# 直接選取要讓模型學習的五個特徵欄位。
features = ['link_count', 'urgent_words', 'exclamation_count', 'sender_known', 'email_type']

# Convert the selected email-type feature into numeric columns.
# 將選取的郵件類型欄位轉成模型可用的數值欄位。
X = pd.get_dummies(df[features], dtype=int)

# Keep spam/not-spam as the correct answer.
# 將是否為垃圾郵件保留為答案。
y = df['is_spam']

# Reserve 20% of the emails as unseen test data.
# 留下兩成郵件，最後才用來測試模型。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Check the number of emails in each group.
# 確認訓練與測試資料的筆數。
print('訓練:', len(X_train), '測試:', len(X_test))
```

In [ ]:
# 在這格輸入上方「練習 1」的程式碼。


### 練習 2｜訓練隨機森林並測試（照範例輸入下一格）

```python
# Create a forest of 60 small decision trees.
# 建立包含六十棵小樹的隨機森林。
model = RandomForestClassifier(n_estimators=60, max_depth=5, random_state=42)

# Let the forest learn from the training emails.
# 使用訓練資料讓模型學習。
model.fit(X_train, y_train)

# Predict the 100 unseen emails.
# 預測一百封測試郵件。
pred = model.predict(X_test)

# Show the test accuracy and number of mistakes.
# 查看測試正確率及猜錯幾封。
print('Accuracy:', round(accuracy_score(y_test, pred) * 100), '%')

# Count all test mistakes.
# 計算測試資料猜錯幾封。
print('猜錯:', int((pred != y_test).sum()), '封')
```

In [ ]:
# 在這格輸入上方「練習 2」的程式碼。


### Optional 1｜三折驗證，比較決策樹與隨機森林（直接執行）

In [ ]:
# Build a small decision tree for comparison.
# 建立一棵決策樹，與隨機森林進行比較。
tree = DecisionTreeClassifier(max_depth=3, random_state=42)

# Rotate three validation groups using only training data.
# 只用訓練資料輪流做三次驗證。
tree_scores = cross_val_score(tree, X_train, y_train, cv=3, scoring='accuracy')

# Evaluate the forest using the same three folds.
# 使用相同的三折驗證評估隨機森林。
forest_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy')

# Compare the average validation accuracies.
# 比較三次驗證的平均正確率。
print('決策樹三折平均:', round(tree_scores.mean() * 100, 1), '%')

# Report the forest's average validation accuracy.
# 顯示隨機森林的平均驗證正確率。
print('隨機森林三折平均:', round(forest_scores.mean() * 100, 1), '%')


### Optional 2｜用 Grid Search 尋找較佳的超參數（直接執行）


In [ ]:
# Specify the forest settings to compare.
# 設定要比較的樹木數量與最大深度。
params = {'n_estimators': [20, 60, 100], 'max_depth': [2, 5, 8]}

# Search nine parameter combinations with three-fold validation on Train.
# 在 Train 上使用三折驗證，比較九組超參數。
search = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=3, scoring='accuracy')

# Run the grid search without using the held-out test set.
# 執行網格搜尋，不使用保留的測試集挑選設定。
search.fit(X_train, y_train)

# Display the best settings and their validation score.
# 顯示驗證表現較佳的設定及平均正確率。
print('最佳參數:', search.best_params_, '三折平均:', round(search.best_score_ * 100, 1), '%')

# Evaluate the selected model once on the reserved test set.
# 最後才使用保留的測試集驗收選出的模型。
print('Test Accuracy:', round(search.score(X_test, y_test) * 100, 1), '%')
